# Streamlined CRISPR Screen Analysis Tutorial

This notebook demonstrates how to run the on-disk analysis workflow provided by ``streamlined_crispr``. It walks through quality control, effect-size estimation via pseudo-bulk summaries, and differential expression testing without ever loading the full matrix into memory.


## Prerequisites

Install the project in editable mode with the optional ``test`` dependencies to obtain AnnData, NumPy, pandas, and SciPy::

    pip install -e .[test]

The commands below assume the repository root is the current working directory.


In [1]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad

from streamlined_crispr import (
    quality_control_summary,
    compute_average_log_expression,
    compute_pseudobulk_expression,
    wald_test,
    wilcoxon_test,
)
from streamlined_crispr.scanpy_validation import compare_with_scanpy


## Create a toy on-disk dataset

To keep the tutorial self-contained we generate a small synthetic AnnData object, assign perturbation labels, and write it to disk as an ``.h5ad`` file. In practice you would point the workflow at an existing file.


In [2]:
data_dir = Path('tutorial_outputs')
data_dir.mkdir(exist_ok=True)
raw_path = data_dir / 'demo_raw.h5ad'

counts = np.array([
    [0, 1, 0, 2, 0],
    [3, 0, 0, 1, 0],
    [0, 2, 1, 0, 0],
    [1, 1, 0, 0, 4],
    [0, 0, 0, 3, 5],
    [2, 0, 0, 1, 6],
    [0, 0, 2, 0, 1],
    [1, 0, 0, 2, 0],
])
obs = pd.DataFrame({
    'perturbation': ['ctrl', 'ctrl', 'KO_TP53', 'KO_TP53', 'KO_BRCA1', 'KO_BRCA1', 'KO_TP53', 'KO_BRCA1']
})
var = pd.DataFrame({
    'gene_symbol': ['GATA3', 'MYC', 'KRAS', 'PTEN', 'BCL2']
})
var.index = var['gene_symbol']
adata = ad.AnnData(counts, obs=obs, var=var)
adata.write(raw_path)
raw_path


/Users/dujinhong/miniforge3/envs/causarray/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


PosixPath('tutorial_outputs/demo_raw.h5ad')

## Run quality control

The :func:`quality_control_summary` function filters low-quality cells, under-powered perturbations, and genes with limited expression. It returns boolean masks and writes a filtered ``.h5ad`` file that subsequent steps can reuse.


In [3]:
qc_result = quality_control_summary(
    raw_path,
    min_genes=1,
    min_cells_per_perturbation=2,
    min_cells_per_gene=2,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbol',
    output_dir=data_dir,
    data_name='demo',
)
qc_result


QualityControlResult(cell_mask=array([ True,  True,  True,  True,  True,  True,  True,  True]), gene_mask=array([ True,  True,  True,  True,  True]), perturbation_keep={'ctrl': True, 'KO_TP53': True, 'KO_BRCA1': True}, filtered_path=PosixPath('tutorial_outputs/demo_filtered.h5ad'), cell_gene_counts=array([2, 2, 2, 3, 2, 3, 2, 2]), gene_cell_counts=array([4, 3, 2, 5, 4]))

In [4]:
filtered_path = qc_result.filtered_path
filtered_path


PosixPath('tutorial_outputs/demo_filtered.h5ad')

We can also inspect how many cells and genes were retained.


In [5]:
qc_summary = {
    'total_cells': qc_result.cell_mask.size,
    'kept_cells': int(qc_result.cell_mask.sum()),
    'total_genes': qc_result.gene_mask.size,
    'kept_genes': int(qc_result.gene_mask.sum()),
}
qc_summary


{'total_cells': 8, 'kept_cells': 8, 'total_genes': 5, 'kept_genes': 5}

## Estimate perturbation effect sizes

After QC we can compute effect sizes using both supported strategies. The functions stream data from disk and write their own ``.h5ad`` artifacts for reproducibility.


In [6]:
avg_effects = compute_average_log_expression(
    filtered_path,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbols',
    output_dir=data_dir,
    data_name='demo_avg',
)
avg_effects


gene_symbols,GATA3,MYC,KRAS,PTEN,BCL2
KO_TP53,-1.988336,1.352055,5.639018,-8.314736,5.639018
KO_BRCA1,0.811518,-4.056014,0.000000,-0.298463,5.848507


In [7]:
pseudo_effects = compute_pseudobulk_expression(
    filtered_path,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbols',
    baseline_count=1.0,
    output_dir=data_dir,
    data_name='demo_pseudo',
)
pseudo_effects


gene_symbols,GATA3,MYC,KRAS,PTEN,BCL2
KO_TP53,-1.908011,0.510586,8.112028,-8.430400,8.112028
KO_BRCA1,-0.705296,-7.419181,0.000000,-0.176237,8.367894


## Differential expression testing

Finally, run the Wald and Wilcoxon tests. Each call returns a dictionary of :class:`DifferentialExpressionResult` objects keyed by perturbation and writes a compact ``.h5ad`` summary.


In [8]:
wald_results = wald_test(
    filtered_path,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbols',
    output_dir=data_dir,
    data_name='demo_wald',
)
{key: value.pvalue[:3] for key, value in wald_results.items()}


{'KO_TP53': array([0.69668873, 0.78221764, 0.04604455]),
 'KO_BRCA1': array([0.87559323, 0.31731051, 1.        ])}

In [9]:
wilcoxon_results = wilcoxon_test(
    filtered_path,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbols',
    output_dir=data_dir,
    data_name='demo_wilcoxon',
)
{key: value.pvalue[:3] for key, value in wilcoxon_results.items()}


{'KO_TP53': array([0.51860502, 0.76709687, 0.1967056 ]),
 'KO_BRCA1': array([0.76709687, 0.22067136, 1.        ])}

## Validate against a Scanpy workflow

The [Scanpy validation plan](../SCANPY_VALIDATION_PLAN.md) outlines the steps we expect to match.
Running the comparison on the Adamson subset dataset confirms that preprocessing, filtering, and pseudo-bulk summaries align within numerical precision.


In [10]:
from pathlib import Path

comparison = compare_with_scanpy(
    Path('../data/Adamson_subset.h5ad'),
    perturbation_column='perturbation',
    control_label='control',
    min_genes=100,
    min_cells_per_perturbation=50,
    min_cells_per_gene=100,
    output_dir=data_dir,
    data_name='adamson_subset',
)
comparison


ComparisonResult(normalization_max_abs_diff=0.0, log1p_max_abs_diff=0.0, streamlined_cell_count=1716, reference_cell_count=1716, streamlined_gene_count=9238, reference_gene_count=9238, avg_log_effect_max_abs_diff=1.3322676295501878e-14, pseudobulk_effect_max_abs_diff=2.6645352591003757e-15, wald_effect_max_abs_diff=0.0, wald_statistic_max_abs_diff=3.2436275887448573e-12, wald_pvalue_max_abs_diff=1.0980105713542798e-13, wilcoxon_effect_max_abs_diff=6.437768240385999e-06, wilcoxon_statistic_max_abs_diff=0.00034650633813654297, wilcoxon_pvalue_max_abs_diff=0.00026923249333310473, streamlined_peak_memory_mb=1105.53125, reference_peak_memory_mb=2445.921875, streamlined_timings={'quality_control': 0.3699375409632921, 'average_log_expression': 0.20485812501283363, 'pseudobulk_expression': 0.09166366700083017, 'wald_test': 0.23908262501936406, 'wilcoxon_test': 1.2590541659737937, 'normalize_total+log1p': 0.28097387496382}, reference_timings={'normalize_total': 0.0226421250263229, 'log1p': 0.12

In [11]:
import pandas as pd

summary = pd.Series({
    'streamlined_cells': comparison.streamlined_cell_count,
    'reference_cells': comparison.reference_cell_count,
    'streamlined_genes': comparison.streamlined_gene_count,
    'reference_genes': comparison.reference_gene_count,
    'normalization_max_abs_diff': comparison.normalization_max_abs_diff,
    'log1p_max_abs_diff': comparison.log1p_max_abs_diff,
    'avg_log_effect_max_abs_diff': comparison.avg_log_effect_max_abs_diff,
    'pseudobulk_effect_max_abs_diff': comparison.pseudobulk_effect_max_abs_diff,
    'wald_effect_max_abs_diff': comparison.wald_effect_max_abs_diff,
    'wald_statistic_max_abs_diff': comparison.wald_statistic_max_abs_diff,
    'wald_pvalue_max_abs_diff': comparison.wald_pvalue_max_abs_diff,
    'wilcoxon_effect_max_abs_diff': comparison.wilcoxon_effect_max_abs_diff,
    'wilcoxon_statistic_max_abs_diff': comparison.wilcoxon_statistic_max_abs_diff,
    'wilcoxon_pvalue_max_abs_diff': comparison.wilcoxon_pvalue_max_abs_diff,
    'streamlined_peak_memory_mb':comparison.streamlined_peak_memory_mb,
    'reference_peak_memory_mb':comparison.reference_peak_memory_mb,
}).to_frame('value')
summary


,value
streamlined_cells,1.716000e+03
reference_cells,1.716000e+03
streamlined_genes,9.238000e+03
reference_genes,9.238000e+03
normalization_max_abs_diff,0.000000e+00
log1p_max_abs_diff,0.000000e+00
avg_log_effect_max_abs_diff,1.332268e-14
pseudobulk_effect_max_abs_diff,2.664535e-15
wald_effect_max_abs_diff,0.000000e+00
wald_statistic_max_abs_diff,3.243628e-12


In [12]:
timings = (
    pd.DataFrame({
        'streamlined_seconds': comparison.streamlined_timings,
        'scanpy_seconds': comparison.reference_timings,
    })
    .sort_index()
)
timings['delta_seconds'] = timings['streamlined_seconds'] - timings['scanpy_seconds']
timings['scanpy_over_streamlined'] = timings['scanpy_seconds'] / timings['streamlined_seconds']
total = pd.DataFrame(
    {
        'streamlined_seconds': [timings['streamlined_seconds'].sum()],
        'scanpy_seconds': [timings['scanpy_seconds'].sum()],
    },
    index=['total'],
)
total['delta_seconds'] = total['streamlined_seconds'] - total['scanpy_seconds']
total['scanpy_over_streamlined'] = total['scanpy_seconds'] / total['streamlined_seconds']
timings = pd.concat([timings, total])
timings


,streamlined_seconds,scanpy_seconds,delta_seconds,scanpy_over_streamlined
average_log_expression,0.204858,NaN,NaN,NaN
filter_cells,NaN,0.016258,NaN,NaN
filter_genes,NaN,0.025667,NaN,NaN
filtered_log1p,NaN,0.102691,NaN,NaN
filtered_normalize_total,NaN,0.020326,NaN,NaN
log1p,NaN,0.128622,NaN,NaN
normalize_total,NaN,0.022642,NaN,NaN
normalize_total+log1p,0.280974,NaN,NaN,NaN
pseudobulk_expression,0.091664,NaN,NaN,NaN
quality_control,0.369938,NaN,NaN,NaN


## Inspect generated files

Every major step leaves behind an ``.h5ad`` file in the output directory. This makes it easy to resume the analysis from any point or inspect intermediate results.


In [13]:
sorted(path.name for path in data_dir.glob('*.h5ad'))


['adamson_subset_avg_log_effects.h5ad',
 'adamson_subset_filtered.h5ad',
 'adamson_subset_pseudobulk_effects.h5ad',
 'adamson_subset_wald_de.h5ad',
 'adamson_subset_wilcoxon_de.h5ad',
 'demo_avg_avg_log_effects.h5ad',
 'demo_filtered.h5ad',
 'demo_pseudo_pseudobulk_effects.h5ad',
 'demo_raw.h5ad',
 'demo_wald_wald_de.h5ad',
 'demo_wilcoxon_wilcoxon_de.h5ad']